In [ ]:
import torch
import torch.nn as nn
from torchvision.models.video import s3d, S3D_Weights
import json
import cv2
import numpy as np
from torch.utils.data import Dataset
import random
from tqdm.auto import tqdm
import time
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast
from transformers import VideoMAEModel, VideoMAEConfig


cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

/home/haod6/.conda/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/haod6/.conda/envs/llm/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
# KINETICS_MEAN = [0.43216, 0.394666, 0.37645]
# KINETICS_STD  = [0.22803, 0.22145,  0.216989]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

In [ ]:
class VideoMAEProbe(nn.Module):
    def __init__(self, num_classes=300):
        super().__init__()
        
        self.backbone = VideoMAEModel.from_pretrained(
            # "MCG-NJU/videomae-base-finetuned-kinetics",
            "CHANGE TO YOUR OWN PATH",
            local_files_only=True
        )

        current_cache = os.environ.get("HF_HOME", "系统默认路径")
        print(f"验证成功：模型权重已离线加载！")
        print(f"依赖的本地缓存目录为: {current_cache}")
        
        for param in self.backbone.parameters():
            param.requires_grad = False
        hidden_size = self.backbone.config.hidden_size  # 768
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(p=0.5),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        # x: [B, C, T, H, W] → VideoMAE 需要 [B, T, C, H, W]
        x = x.permute(0, 2, 1, 3, 4)
        outputs = self.backbone(pixel_values=x)
        features = outputs.last_hidden_state.mean(dim=1)  # [B, 768]
        return self.classifier(features)

In [5]:
def build_linear_probe(network=None, num_classes=300):
    if network is None:
        return None

    if network == "cnn":
        model = s3d(weights=S3D_Weights.DEFAULT)
        for param in model.parameters():
            param.requires_grad = False
        in_ch = model.classifier[1].in_channels
        model.classifier[1] = nn.Sequential(
            nn.Dropout3d(p=0.5),
            nn.Conv3d(in_ch, num_classes, kernel_size=1)
        )
        for param in model.classifier[1].parameters():
            param.requires_grad = True

    elif network == "vit":
        model = VideoMAEProbe(num_classes)

    elif network == "lstm":
        return None

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"[{network.upper()}] Trainable: {trainable:,} / {total:,} "
          f"({100*trainable/total:.2f}%)")
    return model

In [6]:
class WLASLDataset(Dataset):
    def __init__(self, json_file, video_root, split='train',
                 num_frames=32, label_map=None):
        with open(json_file, 'r') as f:
            full_data = json.load(f)

        self.video_root = video_root
        self.num_frames = num_frames
        self.split      = split
        self.video_ids  = []
        self.labels     = []

        # ── label map ──
        if label_map is None:
            all_actions = set()
            for info in full_data.values():
                raw   = info.get('action', info.get('label'))
                a_str = str(raw[0] if isinstance(raw, list) else raw)
                all_actions.add(a_str)
            self.action_to_idx = {a: i for i, a in enumerate(sorted(all_actions))}
        else:
            self.action_to_idx = label_map

        # ── 过滤当前 split ──
        for vid_id, info in full_data.items():
            if info.get('subset') != split:
                continue
            video_path = os.path.join(video_root, f"{vid_id}.mp4")
            if not os.path.exists(video_path):
                continue
            raw   = info.get('action', info.get('label'))
            a_str = str(raw[0] if isinstance(raw, list) else raw)
            if a_str not in self.action_to_idx:
                continue
            self.labels.append(self.action_to_idx[a_str])
            self.video_ids.append(vid_id)

        print(f"[{split.upper()}] {len(self.video_ids)} videos | "
              f"{len(self.action_to_idx)} classes")

    def __len__(self):
        return len(self.video_ids)

    def _load_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total = max(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), 1)
        
        # 用 set 存目标帧号，顺序读取，O(1) 查找
        indices = set(np.linspace(0, total - 1, self.num_frames, dtype=int))
        
        frames = []
        last_good = None
        
        for i in range(total):
            ret, frame = cap.read()
            if not ret:
                break
            if i in indices:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (256, 256))
                frames.append(frame)
                last_good = frame
            if len(frames) == self.num_frames:
                break
        
        cap.release()
        
        if not frames:
            raise ValueError("0 frames read")
        
        while len(frames) < self.num_frames:
            frames.append(last_good)
        
        return frames

    # ── 数据增强（训练 vs 测试不同策略）──
    def _transform(self, frames):
        H, W = frames[0].shape[:2]

        if self.split == 'train':
            # 1. RandomResizedCrop：先 resize 到 256，再随机裁 224
            scale  = random.uniform(0.7, 1.0)
            new_h  = int(H * scale)
            new_w  = int(W * scale)
            top    = random.randint(0, H - new_h)
            left   = random.randint(0, W - new_w)
            frames = [f[top:top+new_h, left:left+new_w] for f in frames]
            frames = [cv2.resize(f, (224, 224)) for f in frames]

            # 2. RandomHorizontalFlip
            if random.random() > 0.5:
                frames = [cv2.flip(f, 1) for f in frames]

        else:
            # 测试：resize 到 256 → CenterCrop 224
            frames = [cv2.resize(f, (256, 256)) for f in frames]
            top, left = 16, 16          # (256-224)//2
            frames = [f[top:top+224, left:left+224] for f in frames]

        # stack → [T, H, W, C] → [C, T, H, W]，归一化
        video = np.stack(frames).astype(np.float32) / 255.0  # [T,H,W,C]
        mean  = np.array(IMAGENET_MEAN, dtype=np.float32)
        std   = np.array(IMAGENET_STD, dtype=np.float32)
        video = (video - mean) / std                          # broadcast
        video = video.transpose(3, 0, 1, 2)                  # [C,T,H,W]
        return torch.from_numpy(video)

    def __getitem__(self, idx):
        try:
            video_path = os.path.join(self.video_root, f"{self.video_ids[idx]}.mp4")
            frames     = self._load_frames(video_path)
            video      = self._transform(frames)
            label      = torch.tensor(self.labels[idx], dtype=torch.long)
            return video, label
        except Exception as e:
            # fallback：换一个随机样本
            return self.__getitem__(random.randint(0, len(self) - 1))

In [ ]:

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 1e-3
WORKERS = 2
NETWORK = "vit"
NUM_FRAMES = 16 # 32 for S3D
NUM_CLASSES = 300
JSON_FILE = ""
VIDEO_ROOT = ""
CHECKPOINT_PATH = ""



In [8]:
def run_epoch(model, loader, optimizer, criterion, scaler, device, epoch, EPOCHS, train=True, network="cnn"):

    avg_lat, fps = 0.0, 0.0 
    total_time = 0.0
    latencies = []
    
    if train:
        model.eval()  
        if network == "cnn":
            model.classifier[1].train()
        elif network == "vit":
            model.classifier.train()
    else:
        model.eval()

    total_loss, correct, total = 0.0, 0, 0
    d = "Train" if train else "Val"
    pbar = tqdm(loader, desc=d+f" [{epoch+1}/{EPOCHS}]", leave=False)

    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)

        t0 = time.perf_counter()
        
        if train:
            optimizer.zero_grad()
            with autocast('cuda'):
                outputs = model(inputs).view(inputs.size(0), -1)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad():
                with autocast('cuda'):
                    outputs = model(inputs).view(inputs.size(0), -1)
                    loss = criterion(outputs, labels)

                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                t1 = time.perf_counter()

                batch_time = t1 - t0
                total_time += batch_time
                latencies.append(batch_time / inputs.size(0) * 1000)  # ms/video

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        pbar.set_postfix(loss=f"{loss.item():.3f}",
                         acc=f"{100.*correct/total:.1f}%")

    avg_loss = total_loss / len(loader)
    acc = 100. * correct / total

    if not train and latencies:
        avg_lat = np.mean(latencies)
        fps = total / total_time
        print(f"Val Acc: {acc:.2f} | Latency: {avg_lat:.2f} ms/video | FPS: {fps:.1f}")

    return avg_loss, acc, avg_lat, fps
        

In [9]:
if __name__ == "__main__":
    # ── 数据集 ──
    train_set = WLASLDataset(JSON_FILE, VIDEO_ROOT, split='train',
                             num_frames=NUM_FRAMES)
    val_set = WLASLDataset(JSON_FILE, VIDEO_ROOT, split='val',
                             num_frames=NUM_FRAMES,
                             label_map=train_set.action_to_idx)
    test_set = WLASLDataset(JSON_FILE, VIDEO_ROOT, split='test',
                             num_frames=NUM_FRAMES,
                             label_map=train_set.action_to_idx)

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=WORKERS, pin_memory=True,
                              persistent_workers=True  )
    val_loader = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=WORKERS, pin_memory=True,
                              persistent_workers=True  )
    test_loader = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=WORKERS, pin_memory=True,
                              persistent_workers=True  )

    print(DEVICE)
    model = build_linear_probe(NETWORK, NUM_CLASSES).to(DEVICE)
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE, weight_decay=1e-4
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = GradScaler('cuda')

    best_val_acc = 0.0

    for epoch in range(EPOCHS):
        t0 = time.time()

        train_loss, train_acc, _, _ = run_epoch(model, train_loader, optimizer, criterion, scaler, DEVICE, epoch, EPOCHS, train=True, network=NETWORK)
        val_loss, val_acc, val_lat, val_fps = run_epoch(model, val_loader, optimizer, criterion, scaler, DEVICE, epoch, EPOCHS, train=False, network=NETWORK)

        scheduler.step(val_acc)

        duration = time.time() - t0
        
        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
              f"Train Loss {train_loss:.4f} Acc {train_acc:.2f}% | "
              f"Val Loss {val_loss:.4f} Acc {val_acc:.2f}% | "
              f"{duration:.1f}s")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch':            epoch,
                'model_state_dict': model.state_dict(),
                'val_acc':          val_acc,
                'label_map':        train_set.action_to_idx,
            }, CHECKPOINT_PATH)
            print(f"✅ Best model saved (val acc {val_acc:.2f}%)")

    print("\n" + "="*50)
    ckpt = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(ckpt['model_state_dict'])
    _, test_acc, test_lat, test_fps = run_epoch(
        model, test_loader, None, criterion, scaler, DEVICE, 0, 1, train=False)
    print(f"Final Test Acc: {test_acc:.2f}% | Test Latency: {test_lat:.2f} | Test FPS: {test_fps:.1f}")

[TRAIN] 1897 videos | 300 classes
[VAL] 446 videos | 300 classes
[TEST] 317 videos | 300 classes
cuda
验证成功：模型权重已离线加载！
依赖的本地缓存目录为: /home/haod6/assignment3/model/vit
[VIT] Trainable: 232,236 / 86,457,900 (0.27%)


Train [1/20]:   0%|          | 0/30 [00:00<?, ?it/s][h264 @ 0x571115637240] Invalid NAL unit size (745 > 472).
[h264 @ 0x571115637240] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x571114f8f380] stream 1, offset 0x3b468: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x571114f8f380] stream 1, offset 0x3b7d3: partial file
[h264 @ 0x571114df6140] Invalid NAL unit size (745 > 472).
[h264 @ 0x571114df6140] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x571113aa3c80] stream 1, offset 0x3b468: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x571113aa3c80] stream 1, offset 0x3b7d3: partial file
Train [1/20]:  17%|█▋        | 5/30 [00:15<00:42,  1.71s/it, acc=0.0%, loss=6.077][h264 @ 0x5711155b9900] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x5711155b9900] missing picture in access unit with size 10780
[h264 @ 0x57111576e4c0] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x57111576e4c0] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 

Val Acc: 4.26 | Latency: 1.47 ms/video | FPS: 680.9
Epoch [01/20] Train Loss 5.9670 Acc 0.95% | Val Loss 5.4479 Acc 4.26% | 67.9s
✅ Best model saved (val acc 4.26%)


Train [2/20]:  53%|█████▎    | 16/30 [00:27<00:15,  1.13s/it, acc=6.2%, loss=5.054][h264 @ 0x571114d92b40] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x571114d92b40] missing picture in access unit with size 10780
[h264 @ 0x571115091880] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x571115091880] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x571114d8a680] stream 1, offset 0x2a27a7: partial file
Train [2/20]:  80%|████████  | 24/30 [00:38<00:06,  1.15s/it, acc=6.6%, loss=4.874][h264 @ 0x5711155be780] Invalid NAL unit size (745 > 472).
[h264 @ 0x5711155be780] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x571113aa3c80] stream 1, offset 0x3b468: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x571113aa3c80] stream 1, offset 0x3b7d3: partial file
                                                                                   

Val Acc: 7.40 | Latency: 1.42 ms/video | FPS: 703.9
Epoch [02/20] Train Loss 5.0404 Acc 7.33% | Val Loss 5.0602 Acc 7.40% | 62.9s
✅ Best model saved (val acc 7.40%)


...

Train [19/20]:  10%|█         | 3/30 [00:10<01:21,  3.01s/it, acc=69.8%, loss=2.163][h264 @ 0x57111553f9c0] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x57111553f9c0] missing picture in access unit with size 10780
[h264 @ 0x571114f59a80] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x571114f59a80] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x5711129c3080] stream 1, offset 0x2a27a7: partial file
Train [19/20]:  40%|████      | 12/30 [00:22<00:23,  1.32s/it, acc=72.7%, loss=2.286][h264 @ 0x57111565a440] Invalid NAL unit size (745 > 472).
[h264 @ 0x57111565a440] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x571113aa3c80] stream 1, offset 0x3b468: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x571113aa3c80] stream 1, offset 0x3b7d3: partial file
                                                                                     

Val Acc: 19.51 | Latency: 1.45 ms/video | FPS: 689.3
Epoch [19/20] Train Loss 2.2354 Acc 69.90% | Val Loss 4.3353 Acc 19.51% | 63.0s
✅ Best model saved (val acc 19.51%)


Train [20/20]:  13%|█▎        | 4/30 [00:11<00:52,  2.03s/it, acc=75.0%, loss=1.944][h264 @ 0x571114d76bc0] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x571114d76bc0] missing picture in access unit with size 10780
[h264 @ 0x571114d61b80] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x571114d61b80] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x571113ab1bc0] stream 1, offset 0x2a27a7: partial file
Train [20/20]:  33%|███▎      | 10/30 [00:19<00:24,  1.24s/it, acc=72.0%, loss=2.357][h264 @ 0x571113b7a2c0] Invalid NAL unit size (745 > 472).
[h264 @ 0x571113b7a2c0] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x571115641200] stream 1, offset 0x3b468: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x571115641200] stream 1, offset 0x3b7d3: partial file
                                                                                     

Val Acc: 19.96 | Latency: 1.49 ms/video | FPS: 672.5
Epoch [20/20] Train Loss 2.2063 Acc 69.74% | Val Loss 4.3341 Acc 19.96% | 64.4s
✅ Best model saved (val acc 19.96%)



Val Acc: 17.98 | Latency: 1.92 ms/video | FPS: 520.6
Final Test Acc: 17.98% | Test Latency: 1.92 | Test FPS: 520.6


In [11]:
if __name__ == "__main__":
    train_set = WLASLDataset(JSON_FILE, VIDEO_ROOT, split='train',
                             num_frames=NUM_FRAMES)
    val_set = WLASLDataset(JSON_FILE, VIDEO_ROOT, split='val',
                             num_frames=NUM_FRAMES,
                             label_map=train_set.action_to_idx)
    test_set = WLASLDataset(JSON_FILE, VIDEO_ROOT, split='test',
                             num_frames=NUM_FRAMES,
                             label_map=train_set.action_to_idx)

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=WORKERS, pin_memory=True,
                              persistent_workers=True)
    val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=WORKERS, pin_memory=True,
                              persistent_workers=True)
    test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=WORKERS, pin_memory=True,
                              persistent_workers=True)

    print(DEVICE)
    model = build_linear_probe(NETWORK, NUM_CLASSES).to(DEVICE)

    # ── 加载已有 checkpoint，继续训练 ──
    ckpt = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(ckpt['model_state_dict'])
    best_val_acc = ckpt['val_acc']
    print(f"▶ Resumed from checkpoint, best val acc so far: {best_val_acc:.2f}%")

    EXTRA_LEARNING_RATE = 3e-4
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=EXTRA_LEARNING_RATE, weight_decay=1e-4
    )
    scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)
    criterion  = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler     = GradScaler('cuda')

    EXTRA_EPOCHS = 10  # 继续跑的轮数

    for epoch in range(EXTRA_EPOCHS):
        t0 = time.time()
        train_loss, train_acc, _, _ = run_epoch(
            model, train_loader, optimizer, criterion, scaler,
            DEVICE, epoch, EXTRA_EPOCHS, train=True, network=NETWORK)
        val_loss, val_acc, val_lat, val_fps = run_epoch(
            model, val_loader, optimizer, criterion, scaler,
            DEVICE, epoch, EXTRA_EPOCHS, train=False, network=NETWORK)

        scheduler.step(val_acc)
        duration = time.time() - t0

        print(f"Epoch [{epoch+1:02d}/{EXTRA_EPOCHS}] "
              f"Train Loss {train_loss:.4f} Acc {train_acc:.2f}% | "
              f"Val Loss {val_loss:.4f} Acc {val_acc:.2f}% | "
              f"{duration:.1f}s")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch':            epoch,
                'model_state_dict': model.state_dict(),
                'val_acc':          val_acc,
                'label_map':        train_set.action_to_idx,
            }, CHECKPOINT_PATH)
            print(f"✅ Best model saved (val acc {val_acc:.2f}%)")

    print("\n" + "="*50)
    ckpt = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(ckpt['model_state_dict'])
    _, test_acc, test_lat, test_fps = run_epoch(
        model, test_loader, None, criterion, scaler, DEVICE, 0, 1, train=False)
    print(f"Final Test Acc for EXTRA {EXTRA_EPOCHS} EPOCHS: {test_acc:.2f}% | Test Latency: {test_lat:.2f} | Test FPS: {test_fps:.1f}")

[TRAIN] 1897 videos | 300 classes
[VAL] 446 videos | 300 classes
[TEST] 317 videos | 300 classes
cuda
验证成功：模型权重已离线加载！
依赖的本地缓存目录为: /home/haod6/assignment3/model/vit
[VIT] Trainable: 232,236 / 86,457,900 (0.27%)
▶ Resumed from checkpoint, best val acc so far: 19.96%


Train [1/10]:  53%|█████▎    | 16/30 [00:29<00:16,  1.16s/it, acc=70.1%, loss=2.116][h264 @ 0x5711136c5a80] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x5711136c5a80] missing picture in access unit with size 10780
[h264 @ 0x57113d639a80] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x57113d639a80] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x57113d8f5b40] stream 1, offset 0x2a27a7: partial file
Train [1/10]:  60%|██████    | 18/30 [00:32<00:13,  1.14s/it, acc=69.8%, loss=2.351][h264 @ 0x57113d7984c0] Invalid NAL unit size (745 > 472).
[h264 @ 0x57113d7984c0] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x57113d42eb80] stream 1, offset 0x3b468: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x57113d42eb80] stream 1, offset 0x3b7d3: partial file
                                                                                    

Val Acc: 18.39 | Latency: 1.48 ms/video | FPS: 673.8
Epoch [01/10] Train Loss 2.1670 Acc 70.74% | Val Loss 4.3160 Acc 18.39% | 66.3s


Train [2/10]:   0%|          | 0/30 [00:00<?, ?it/s][h264 @ 0x57113d5f7cc0] Invalid NAL unit size (745 > 472).
[h264 @ 0x57113d5f7cc0] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x57113d42eb80] stream 1, offset 0x3b468: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x57113d42eb80] stream 1, offset 0x3b7d3: partial file
Train [2/10]:  27%|██▋       | 8/30 [00:17<00:31,  1.41s/it, acc=76.2%, loss=2.144][h264 @ 0x57113d6ae9c0] Invalid NAL unit size (745 > 472).
[h264 @ 0x57113d6ae9c0] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x57113d8f5b40] stream 1, offset 0x3b468: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x57113d8f5b40] stream 1, offset 0x3b7d3: partial file
Train [2/10]:  67%|██████▋   | 20/30 [00:34<00:13,  1.37s/it, acc=73.4%, loss=2.022][h264 @ 0x5711136c5a80] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x5711136c5a80] missing picture in access unit with size 10780
[h264 @ 0x571140bbc9c0] Invalid NAL unit size (71678 > 10776).
[h264 

Val Acc: 19.96 | Latency: 1.43 ms/video | FPS: 698.3
Epoch [02/10] Train Loss 2.1377 Acc 72.85% | Val Loss 4.2924 Acc 19.96% | 64.7s


...

Train [9/10]:   0%|          | 0/30 [00:00<?, ?it/s][h264 @ 0x5711128d7b80] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x5711128d7b80] missing picture in access unit with size 10780
[h264 @ 0x57113d7b2180] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x57113d7b2180] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x57113d8f5b40] stream 1, offset 0x2a27a7: partial file
Train [9/10]:  27%|██▋       | 8/30 [00:17<00:28,  1.30s/it, acc=78.3%, loss=1.900][h264 @ 0x57113d63be00] Invalid NAL unit size (745 > 472).
[h264 @ 0x57113d63be00] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x57113d8f5b40] stream 1, offset 0x3b468: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x57113d8f5b40] stream 1, offset 0x3b7d3: partial file
                                                                                    

Val Acc: 19.73 | Latency: 1.48 ms/video | FPS: 676.7
Epoch [09/10] Train Loss 2.0473 Acc 75.96% | Val Loss 4.2879 Acc 19.73% | 65.6s


Train [10/10]:   7%|▋         | 2/30 [00:09<01:49,  3.90s/it, acc=77.3%, loss=1.993][h264 @ 0x57113d51b800] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x57113d51b800] missing picture in access unit with size 10780
[h264 @ 0x57113d7130c0] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x57113d7130c0] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x57113d8f5b40] stream 1, offset 0x2a27a7: partial file
Train [10/10]:  93%|█████████▎| 28/30 [00:46<00:02,  1.19s/it, acc=76.6%, loss=2.071][h264 @ 0x57113d711a80] Invalid NAL unit size (745 > 472).
[h264 @ 0x57113d711a80] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x57113d8f5b40] stream 1, offset 0x3b468: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x57113d8f5b40] stream 1, offset 0x3b7d3: partial file
                                                                                     

Val Acc: 19.73 | Latency: 1.60 ms/video | FPS: 626.3
Epoch [10/10] Train Loss 2.0120 Acc 76.86% | Val Loss 4.2815 Acc 19.73% | 66.3s



Val Acc: 18.30 | Latency: 1.56 ms/video | FPS: 640.3
Final Test Acc for EXTRA 10 EPOCHS: 18.30% | Test Latency: 1.56 | Test FPS: 640.3
